# Between-Subjects Aggregation
Loads per-subject results saved by each analysis notebook and averages across participants.

In [1]:
from pathlib import Path
import numpy as np

PROJECT_ROOT    = Path('..').resolve()
DERIVATIVES_DIR = PROJECT_ROOT / 'data' / 'derivatives' / 'mne_preproc'
TASK            = 'Cannonball MF'

# Add subjects here as data becomes available
SUBJECTS = ['01']
SESSION  = '01'

def load_subject(subject):
    d = DERIVATIVES_DIR / f'sub-{subject}' / f'ses-{SESSION}'
    prefix = f'sub-{subject}_ses-{SESSION}_task-{TASK}'
    return {
        'main'     : np.load(str(d / f'{prefix}_results.npz'),          allow_pickle=True),
        'rw_alpha' : np.load(str(d / f'{prefix}_rw_alpha_results.npz'), allow_pickle=True),
        'rw_beta'  : np.load(str(d / f'{prefix}_rw_beta_results.npz'),  allow_pickle=True),
        'tgm'      : np.load(str(d / f'{prefix}_tgm_results.npz'),      allow_pickle=True),
    }

data = {sub: load_subject(sub) for sub in SUBJECTS}
print(f'Loaded {len(data)} subject(s): {SUBJECTS}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/arthurhsia/Desktop/Psychology/EEG/data/derivatives/mne_preproc/sub-01/ses-01/sub-01_ses-01_task-Cannonball MF_results.npz'

In [ ]:
import pandas as pd

# --- Alpha / Beta correlations with V(t) and |PE| ---
rows = []
for sub, d in data.items():
    m = d['main']
    rows.append({
        'subject'    : sub,
        'r_alpha_V'  : float(m['r_alpha_V']),
        'p_alpha_V'  : float(m['p_alpha_V']),
        'r_beta_V'   : float(m['r_beta_V']),
        'p_beta_V'   : float(m['p_beta_V']),
        'r_alpha_PE' : float(m['r_alpha_PE']),
        'p_alpha_PE' : float(m['p_alpha_PE']),
        'r_beta_PE'  : float(m['r_beta_PE']),
        'p_beta_PE'  : float(m['p_beta_PE']),
    })

corr_df = pd.DataFrame(rows).set_index('subject')
print('=== Correlation summary ===')
print(corr_df.to_string())

if len(SUBJECTS) > 1:
    print('\n=== Group means ===')
    print(corr_df.mean().to_string())

In [ ]:
# --- RW model fit: best learning rate and peak r ---
rw_rows = []
for sub, d in data.items():
    ra = d['rw_alpha']
    rb = d['rw_beta']
    rw_rows.append({
        'subject'        : sub,
        'alpha_best_lr'  : float(ra['best_alpha']),
        'alpha_best_r'   : float(ra['best_r']),
        'alpha_best_p'   : float(ra['best_p']),
        'beta_best_lr'   : float(rb['best_alpha']),
        'beta_best_r'    : float(rb['best_r']),
        'beta_best_p'    : float(rb['best_p']),
    })

rw_df = pd.DataFrame(rw_rows).set_index('subject')
print('=== RW model fit summary ===')
print(rw_df.to_string())

In [ ]:
import matplotlib.pyplot as plt

# --- Group-averaged r vs alpha curves ---
alpha_grid = data[SUBJECTS[0]]['rw_alpha']['alpha_grid']

alpha_curves = np.stack([data[s]['rw_alpha']['r_curve'] for s in SUBJECTS])
beta_curves  = np.stack([data[s]['rw_beta']['r_curve']  for s in SUBJECTS])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, curves, label in zip(axes, [alpha_curves, beta_curves], ['Alpha (8–13 Hz)', 'Beta (13–30 Hz)']):
    for i, s in enumerate(SUBJECTS):
        ax.plot(alpha_grid, curves[i], alpha=0.4, color='steelblue', label=f'sub-{s}')
    if len(SUBJECTS) > 1:
        ax.plot(alpha_grid, curves.mean(axis=0), color='navy', linewidth=2, label='Mean')
    ax.axhline(0, color='k', linewidth=0.5, linestyle='--')
    ax.set_xlabel('Learning rate α')
    ax.set_ylabel('Pearson r')
    ax.set_title(f'{label}  |  r vs α')
    ax.legend(fontsize=8)

fig.suptitle('RW model fit: r vs learning rate')
plt.tight_layout()
plt.show()

In [ ]:
# --- Group-averaged TGM ---
tgm_stack = np.stack([data[s]['tgm']['scores_mean'] for s in SUBJECTS])  # (n_subjects, n_train, n_test)
tgm_mean  = tgm_stack.mean(axis=0)
times     = data[SUBJECTS[0]]['tgm']['times']

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    tgm_mean,
    origin='lower',
    extent=[times[0], times[-1], times[0], times[-1]],
    aspect='auto',
    vmin=0.4, vmax=0.6,
    cmap='RdBu_r'
)
fig.colorbar(im, ax=ax, label='AUC (reward vs loss)')
ax.axhline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
ax.axvline(0, color='k', linestyle='--', linewidth=0.8, alpha=0.5)
ax.plot(times, times, color='gray', linewidth=0.6, linestyle=':')
ax.set_xlabel('Test time (s relative to outcome)')
ax.set_ylabel('Train time (s relative to outcome)')
ax.set_title(f'Group TGM  |  N={len(SUBJECTS)}  |  SVM reward vs loss')
plt.tight_layout()
plt.show()